## Импорт библиотек и загрузка данных

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('ggplot')
import numpy as np

In [ ]:
import os
import urllib.request

os.makedirs('data', exist_ok=True)

# Скачиваем файлы из репозитория, если их нет локально
files = {
    'data/orders.xlsx': 'https://raw.githubusercontent.com/EvgenySklyarov81/ad-hoc-sales-queries/main/data/orders.xlsx',
    'data/products.xlsx': 'https://raw.githubusercontent.com/EvgenySklyarov81/ad-hoc-sales-queries/main/data/products.xlsx',
}

for path, url in files.items():
    if not os.path.exists(path):
        print(f'Скачиваю {path}...')
        urllib.request.urlretrieve(url, path)
    else:
        print(f'{path} уже существует')

print('Данные готовы.')

In [ ]:
orders = pd.read_excel('data/orders.xlsx')
products = pd.read_excel('data/products.xlsx')
products.columns = ['product_id', 'category', 'sub_category', 'name']

## Знакомство с данными

### Датасет 1, заказы

In [ ]:
orders.info()
orders.head()

### Датасет 2, товары

In [ ]:
products.info()
products.head()

# Самая ходовая товарная группа  
### По какой категории товаров продано больше всего позиций?

In [ ]:
# при соединении отбрасываем товары, отсутствующие в справочнике товаров т.к. нам необходимы названия категорий для каждого товара в продажах
df = orders.merge(products)

In [ ]:
df.sample(3)

In [ ]:
qty_by_category = df.groupby(['category'])['quantity'].agg('sum').reset_index().sort_values('quantity')
qty_by_category

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh('category', 'quantity', data=qty_by_category)
ax.set_xlabel('Количество, шт')
ax.set_title('Продажи товаров по категориям');

# Распределение продаж по подкатегориям
### Оценить распределение количества проданных позиций в каждой товарной категории по подкатегориям

In [ ]:
cat_sub_cat = df.groupby(['category', 'sub_category'])['quantity'].agg('sum').reset_index().sort_values(['category', 'quantity', 'sub_category'], ascending=[True, False, True])
cat_sub_cat

# Найти средний чек в заданную дату
### Какой средний чек был 13.01.2022?

In [ ]:
orders['total'] = orders['quantity'] * orders['price']

In [ ]:
# располагаем данными о продажах только за один день
orders['accepted_at'].min(), orders['accepted_at'].max()

In [ ]:
avg_bill = round(orders.groupby('order_id')['total'].agg('sum').mean(), 2)
print(f'Средний чек за 13.01.2022: {avg_bill}')

# Доля промо в заданной категории
### Какую долю от общих продаж категории Сыры занимают промо (в штуках)

In [ ]:
cheese = df[df['category'] == 'Сыры']

# все продажи сыров и промо-продажи сыров в штуках
regular = cheese[cheese['price'] == cheese['regular_price']]['quantity'].sum()
promo = cheese[cheese['price'] != cheese['regular_price']]['quantity'].sum()

data = [regular, promo]
labels = ['Базовая цена', 'Промо-цена']

fig, ax = plt.subplots(figsize=(10, 8))
ax.pie(data, labels=labels, autopct='%1.0f%%')
ax.set_title('Доля промо-продаж в категории "Сыры", в штуках');

# Посчитать маржу по категориям
* В рублях
* В %

In [ ]:
cat_margin = df.copy()
cat_margin['revenue'] = cat_margin['quantity'] * cat_margin['price']
cat_margin['margin'] = cat_margin['quantity'] * (cat_margin['price'] - cat_margin['cost_price'])

In [ ]:
cat_margin = cat_margin.groupby('category')[['revenue', 'margin']].agg('sum')

In [ ]:
cat_margin['marginality'] = round(cat_margin['margin'] / cat_margin['revenue'] * 100, 2)

In [ ]:
cat_margin = cat_margin.drop(columns='revenue')

In [ ]:
category = cat_margin.sort_values('margin').index
margin = cat_margin['margin'].sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(category, margin)
ax.set_title('Маржа категорий товаров')
ax.set_xlabel('Рублей');

In [ ]:
category = cat_margin.sort_values('marginality').index
marginality = cat_margin['marginality'].sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(category, marginality)
ax.set_title('Маржинальность категорий товаров')
ax.set_xlabel('Процентов');

# ABC анализ по подкатегориям
* ABC-анализ продаж по количеству
* ABC-анализ по сумме продаж

In [ ]:
dataset = df.assign(revenue=df['quantity'] * df['price'])

In [ ]:
dataset = dataset.groupby('sub_category')[['quantity', 'revenue']].agg('sum')

In [ ]:
total_qty = dataset['quantity'].sum()
abc_qty = dataset['quantity'].sort_values(ascending=False).reset_index()
abc_qty['cum_qty'] = abc_qty['quantity'].cumsum()
abc_qty['ratio'] = abc_qty['cum_qty'] / total_qty
abc_qty['qty_group'] = np.where(abc_qty['ratio'] <= 0.8, 'A', np.where(abc_qty['ratio'] <= 0.95, 'B', 'C'))
abc_qty = abc_qty[['sub_category', 'qty_group']]

In [ ]:
total_rev = dataset['revenue'].sum()
abc_rev = dataset['revenue'].sort_values(ascending=False).reset_index()
abc_rev['cum_revenue'] = abc_rev['revenue'].cumsum()
abc_rev['ratio'] = abc_rev['cum_revenue'] / total_rev
abc_rev['rev_group'] = np.where(abc_rev['ratio'] <= 0.8, 'A', np.where(abc_rev['ratio'] <= 0.95, 'B', 'C'))
abc_rev = abc_rev[['sub_category', 'rev_group']]

In [ ]:
abc_result = abc_qty.merge(abc_rev)
abc_result = (
    abc_result.assign(group=abc_result['qty_group'] + abc_result['rev_group'])
    .drop(columns=['qty_group', 'rev_group'])
    .sort_values(['group', 'sub_category'])
    .reset_index(drop=True)
)
abc_result